# Seq2Seq + Bahdanau Attention (최적화 v2 - SentencePiece 적용)
## 영어→한국어 기계번역 | Korean-English TED Talks

### v2 핵심 변경: 토크나이저 전면 교체
- **기존**: Mecab 형태소 분석 (한국어) + 공백 분리 (영어)
  - 문제: 과도한 분절, OOV 다수, 어휘 크기 ~35K/~39K
- **변경**: **SentencePiece BPE** (영어 16K + 한국어 16K)
  - OOV 문제 해소 (서브워드로 분해)
  - 어휘 크기 대폭 감소 → 학습 효율 향상
  - 영어/한국어 모두 데이터 기반 토큰화로 일관성 확보


# 1. 환경 설정

In [ ]:
!pip install datasets sentencepiece -q
!pip install evaluate rouge_score sacrebleu -q

In [ ]:
import re
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math
import sentencepiece as spm
from collections import Counter
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

---
# 2. SentencePiece 토크나이저 로드

### 사전 학습 필요
```bash
# 먼저 별도 스크립트로 SentencePiece 모델을 학습해야 합니다:
python train_sentencepiece.py
```

학습이 완료되면 `en_spm.model`, `ko_spm.model` 파일이 생성됩니다.

In [ ]:
# ============================================================
# 데이터 로드
# ============================================================
dataset = load_dataset("msarmi9/korean-english-multitarget-ted-talks-task")
num_samples = 88000

en_sentences = []
ko_sentences = []

for i, data in enumerate(dataset['train']):
    if i >= num_samples:
        break
    en_sentences.append(data['english'].strip())
    ko_sentences.append(data['korean'].strip())

print(f"영어 문장 수: {len(en_sentences)}")
print(f"한국어 문장 수: {len(ko_sentences)}")

In [ ]:
# ============================================================
# 사전 학습된 SentencePiece 모델 로드
# (python train_sentencepiece.py 로 미리 학습 필요)
# ============================================================
sp_en = spm.SentencePieceProcessor()
sp_ko = spm.SentencePieceProcessor()
sp_en.load('en_spm.model')
sp_ko.load('ko_spm.model')

src_vocab_size = sp_en.get_piece_size()
tar_vocab_size = sp_ko.get_piece_size()

print(f"영어 어휘 크기: {src_vocab_size}")
print(f"한국어 어휘 크기: {tar_vocab_size}")

# 토큰화 테스트
test_en = "Have you had dinner?"
test_ko = "저녁은 드셨나요?"

print(f"\n--- 토큰화 테스트 ---")
print(f"영어 원문: {test_en}")
print(f"  토큰: {sp_en.encode(test_en, out_type=str)}")
print(f"  정수: {sp_en.encode(test_en, out_type=int)}")

print(f"\n한국어 원문: {test_ko}")
print(f"  토큰: {sp_ko.encode(test_ko, out_type=str)}")
print(f"  정수: {sp_ko.encode(test_ko, out_type=int)}")

# 특수 토큰 확인
print(f"\n--- 특수 토큰 ---")
print(f"PAD id: {sp_en.pad_id()} → '{sp_en.id_to_piece(sp_en.pad_id())}'")
print(f"UNK id: {sp_en.unk_id()} → '{sp_en.id_to_piece(sp_en.unk_id())}'")
print(f"BOS id: {sp_en.bos_id()} → '{sp_en.id_to_piece(sp_en.bos_id())}'")
print(f"EOS id: {sp_en.eos_id()} → '{sp_en.id_to_piece(sp_en.eos_id())}'")

---
# 3. 데이터 전처리 (SentencePiece 기반)

### 기존 전처리 파이프라인 vs 새로운 파이프라인
```
[기존]
원문 → regex 필터 → Mecab morphs → 수동 vocab 구축 → 정수 인코딩 → 패딩

[새로운 - SentencePiece]
원문 → 소문자 변환(영어) → SentencePiece encode → 패딩
(vocab 구축, 정수 인코딩이 SentencePiece 내부에서 자동 처리)
```


In [ ]:
# ============================================================
# SentencePiece 기반 데이터 전처리
# - 기존의 복잡한 regex + Mecab 파이프라인을 SentencePiece로 대체
# - <sos>, <eos>는 SentencePiece의 bos_id, eos_id 사용
# ============================================================

def preprocess_and_encode():
    encoder_input = []    # 영어 (소스)
    decoder_input = []    # 한국어 + <sos> (디코더 입력)
    decoder_target = []   # 한국어 + <eos> (디코더 레이블)

    for i in tqdm(range(num_samples), desc="전처리 중"):
        eng = en_sentences[i].lower().strip()  # 영어는 소문자 변환만
        kor = ko_sentences[i].strip()

        # SentencePiece 인코딩 (정수 시퀀스)
        enc_ids = sp_en.encode(eng, out_type=int)
        dec_ids = sp_ko.encode(kor, out_type=int)

        # 디코더 입력: <sos> + 토큰
        dec_input_ids = [sp_ko.bos_id()] + dec_ids
        # 디코더 레이블: 토큰 + <eos>
        dec_target_ids = dec_ids + [sp_ko.eos_id()]

        encoder_input.append(enc_ids)
        decoder_input.append(dec_input_ids)
        decoder_target.append(dec_target_ids)

    return encoder_input, decoder_input, decoder_target

encoder_input, decoder_input, decoder_target = preprocess_and_encode()

# 길이 통계
enc_lens = [len(s) for s in encoder_input]
dec_lens = [len(s) for s in decoder_input]
print(f"\n인코더 입력 길이 - 평균: {np.mean(enc_lens):.1f}, 최대: {max(enc_lens)}, 중간: {np.median(enc_lens):.0f}")
print(f"디코더 입력 길이 - 평균: {np.mean(dec_lens):.1f}, 최대: {max(dec_lens)}, 중간: {np.median(dec_lens):.0f}")

In [ ]:
# ============================================================
# 패딩 및 데이터 분할
# ============================================================
def pad_sequences(sentences, max_len=None):
    if max_len is None:
        max_len = max([len(s) for s in sentences])
    features = np.zeros((len(sentences), max_len), dtype=int)
    for i, s in enumerate(sentences):
        length = min(len(s), max_len)
        features[i, :length] = np.array(s[:length])
    return features

encoder_input = pad_sequences(encoder_input)
decoder_input = pad_sequences(decoder_input)
decoder_target = pad_sequences(decoder_target)

print(f"인코더 입력: {encoder_input.shape}")
print(f"디코더 입력: {decoder_input.shape}")
print(f"디코더 레이블: {decoder_target.shape}")

# 셔플
indices = np.arange(encoder_input.shape[0])
np.random.seed(42)
np.random.shuffle(indices)
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]

# 분할 (90% 훈련, 10% 검증)
n_of_val = int(num_samples * 0.1)
encoder_input_train, encoder_input_test = encoder_input[:-n_of_val], encoder_input[-n_of_val:]
decoder_input_train, decoder_input_test = decoder_input[:-n_of_val], decoder_input[-n_of_val:]
decoder_target_train, decoder_target_test = decoder_target[:-n_of_val], decoder_target[-n_of_val:]

print(f"\n훈련: {encoder_input_train.shape[0]}개, 검증: {encoder_input_test.shape[0]}개")

In [ ]:
# ============================================================
# 인코딩/디코딩 확인
# ============================================================
idx = 0
print("--- 샘플 확인 ---")
print(f"인코더 입력 (정수): {encoder_input_train[idx][:20]}...")

# 영어 복원
enc_ids = [int(x) for x in encoder_input_train[idx] if x != 0]
print(f"영어 복원: {sp_en.decode(enc_ids)}")

# 한국어 복원 (디코더 입력에서 <sos> 제거)
dec_ids = [int(x) for x in decoder_input_train[idx] if x != 0 and x != sp_ko.bos_id()]
print(f"한국어 복원: {sp_ko.decode(dec_ids)}")

---
# 4. 모델 정의 (Seq2Seq + Bahdanau Attention)

### 최적화 사항
1. **SentencePiece BPE** → 어휘 크기 ~39K → 16K, OOV 해소
2. **Bahdanau Additive Attention** → 학습 가능한 가중치로 풍부한 어텐션
3. **2층 Bidirectional LSTM** + 드롭아웃 → 인코더 표현력 강화
4. **출력층: hidden + context + embedding 결합** → 정보 흐름 극대화
5. **Label Smoothing + AdamW + Warmup/Cosine LR + Gradient Clipping + Early Stopping**


In [ ]:
# 하이퍼파라미터
embedding_dim = 512
hidden_units = 512
n_layers = 2
enc_dropout = 0.3
dec_dropout = 0.3

In [ ]:
# ============================================================
# Bahdanau (Additive) Attention
# score = V * tanh(W1 * encoder_output + W2 * decoder_hidden)
# ============================================================
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_units):
        super(BahdanauAttention, self).__init__()
        self.W1 = nn.Linear(hidden_units, hidden_units, bias=False)
        self.W2 = nn.Linear(hidden_units, hidden_units, bias=False)
        self.V = nn.Linear(hidden_units, 1, bias=False)

    def forward(self, encoder_outputs, hidden, mask=None):
        # encoder_outputs: (batch, src_len, hidden)
        # hidden: (1, batch, hidden) → (batch, 1, hidden)
        hidden_expanded = hidden.transpose(0, 1)
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden_expanded))
        attention_scores = self.V(energy)  # (batch, src_len, 1)

        if mask is not None:
            attention_scores = attention_scores.squeeze(2)
            attention_scores = attention_scores.masked_fill(mask, -1e9)
            attention_scores = attention_scores.unsqueeze(2)

        attention_weights = torch.softmax(attention_scores, dim=1)
        context = torch.bmm(attention_weights.transpose(1, 2), encoder_outputs)
        return context, attention_weights

In [ ]:
# ============================================================
# Encoder: 2층 Bidirectional LSTM
# ============================================================
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_units, n_layers, dropout):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_units,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.fc_hidden = nn.Linear(hidden_units * 2, hidden_units)
        self.fc_cell = nn.Linear(hidden_units * 2, hidden_units)
        self.fc_outputs = nn.Linear(hidden_units * 2, hidden_units)

    def forward(self, x):
        x = self.emb_dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(x)
        outputs = self.fc_outputs(outputs)

        # 양방향 결합: forward/backward hidden 합치기
        hidden = self.fc_hidden(torch.cat([hidden[0::2], hidden[1::2]], dim=2))
        cell = self.fc_cell(torch.cat([cell[0::2], cell[1::2]], dim=2))
        return outputs, hidden, cell

In [ ]:
# ============================================================
# Decoder: Bahdanau Attention + 결합 FC + 출력 결합
# ============================================================
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_units, n_layers, dropout):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)
        self.attention = BahdanauAttention(hidden_units)
        self.input_combine = nn.Linear(embedding_dim + hidden_units, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_units,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        # 출력: hidden + context + embedded 결합
        self.fc_out = nn.Linear(hidden_units + hidden_units + embedding_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_outputs, hidden, cell, mask=None):
        embedded = self.emb_dropout(self.embedding(x))
        if embedded.dim() == 2:
            embedded = embedded.unsqueeze(1)

        context, attn_weights = self.attention(encoder_outputs, hidden[-1:], mask)
        seq_len = embedded.shape[1]
        context_repeated = context.repeat(1, seq_len, 1)

        combined_input = torch.relu(
            self.input_combine(torch.cat([embedded, context_repeated], dim=2))
        )
        output, (hidden, cell) = self.lstm(combined_input, (hidden, cell))
        output = self.fc_out(
            self.dropout(torch.cat([output, context_repeated, embedded], dim=2))
        )
        return output, hidden, cell

In [ ]:
# ============================================================
# Seq2Seq 결합
# ============================================================
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        mask = (src == 0).to(src.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        output, _, _ = self.decoder(trg, encoder_outputs, hidden, cell, mask)
        return output

encoder = Encoder(src_vocab_size, embedding_dim, hidden_units, n_layers, enc_dropout)
decoder = Decoder(tar_vocab_size, embedding_dim, hidden_units, n_layers, dec_dropout)
model = Seq2Seq(encoder, decoder)

total_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 수: {total_params:,}")
print(f"\n모델 구조:\n{model}")

---
# 5. 학습 설정 및 훈련

In [ ]:
# Label Smoothing Loss
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, padding_idx=0, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.smoothing = smoothing
        self.vocab_size = vocab_size
        self.padding_idx = padding_idx
        self.criterion = nn.KLDivLoss(reduction='batchmean')

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.vocab_size - 2))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
            true_dist[:, self.padding_idx] = 0
            mask = (target == self.padding_idx)
            true_dist[mask] = 0
        return self.criterion(pred, true_dist)

loss_function = LabelSmoothingLoss(tar_vocab_size, padding_idx=0, smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)

num_epochs = 20
warmup_epochs = 3
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs),
        optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
    ],
    milestones=[warmup_epochs]
)

print(f"에폭: {num_epochs}, Warmup: {warmup_epochs}")
print(f"LR: 0.001, Label Smoothing: 0.1, Grad Clip: 1.0")

In [ ]:
# 데이터로더 생성
batch_size = 256

train_dataset = TensorDataset(
    torch.tensor(encoder_input_train, dtype=torch.long),
    torch.tensor(decoder_input_train, dtype=torch.long),
    torch.tensor(decoder_target_train, dtype=torch.long)
)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = TensorDataset(
    torch.tensor(encoder_input_test, dtype=torch.long),
    torch.tensor(decoder_input_test, dtype=torch.long),
    torch.tensor(decoder_target_test, dtype=torch.long)
)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

model.to(device)
print(f"훈련 배치: {len(train_dataloader)}, 검증 배치: {len(valid_dataloader)}")

In [ ]:
def evaluation(model, dataloader, loss_function, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.no_grad():
        for enc_in, dec_in, dec_tar in dataloader:
            enc_in, dec_in, dec_tar = enc_in.to(device), dec_in.to(device), dec_tar.to(device)
            outputs = model(enc_in, dec_in, teacher_forcing_ratio=0.0)
            loss = loss_function(outputs.view(-1, outputs.size(-1)), dec_tar.view(-1))
            total_loss += loss.item()
            mask = dec_tar != 0
            total_correct += ((outputs.argmax(dim=-1) == dec_tar) * mask).sum().item()
            total_count += mask.sum().item()
    return total_loss / len(dataloader), total_correct / total_count

In [ ]:
# ============================================================
# Training Loop
# ============================================================
best_val_loss = float('inf')
patience = 5
patience_counter = 0
train_losses_log, valid_losses_log, valid_accs_log, lr_log = [], [], [], []

for epoch in range(num_epochs):
    model.train()
    current_tf_ratio = max(0.0, 1.0 - (epoch / num_epochs))

    for enc_in, dec_in, dec_tar in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        enc_in, dec_in, dec_tar = enc_in.to(device), dec_in.to(device), dec_tar.to(device)
        optimizer.zero_grad()
        outputs = model(enc_in, dec_in, teacher_forcing_ratio=current_tf_ratio)
        loss = loss_function(outputs.view(-1, outputs.size(-1)), dec_tar.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    train_loss, train_acc = evaluation(model, train_dataloader, loss_function, device)
    valid_loss, valid_acc = evaluation(model, valid_dataloader, loss_function, device)

    train_losses_log.append(train_loss)
    valid_losses_log.append(valid_loss)
    valid_accs_log.append(valid_acc)
    lr_log.append(current_lr)

    print(f'Epoch {epoch+1}/{num_epochs} | TF: {current_tf_ratio:.2f} | LR: {current_lr:.6f}')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}')

    if valid_loss < best_val_loss:
        print(f'  ✅ Improved: {best_val_loss:.4f} → {valid_loss:.4f}')
        best_val_loss = valid_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'seq2seq_v2_best.pth')
    else:
        patience_counter += 1
        print(f'  ⚠️ No improvement ({patience_counter}/{patience})')
        if patience_counter >= patience:
            print(f'\n🛑 Early Stopping at epoch {epoch+1}!')
            break

print(f'\n학습 완료! Best Valid Loss: {best_val_loss:.4f}')

In [ ]:
# 학습 곡선 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(train_losses_log, label='Train', marker='o')
axes[0].plot(valid_losses_log, label='Valid', marker='s')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(valid_accs_log, label='Valid Acc', marker='o', color='green')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
axes[2].plot(lr_log, label='LR', marker='o', color='red')
axes[2].set_title('Learning Rate'); axes[2].legend(); axes[2].grid(True)
plt.tight_layout(); plt.savefig('seq2seq_v2_training.png', dpi=150); plt.show()

---
# 6. 추론 및 평가

In [ ]:
# 모델 로드
model.load_state_dict(torch.load('seq2seq_v2_best.pth'))
model.to(device)
val_loss, val_acc = evaluation(model, valid_dataloader, loss_function, device)
print(f'Best model - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

In [ ]:
# ============================================================
# 추론 함수 (SentencePiece 기반)
# ============================================================
def decode_sequence(input_seq, model, max_output_len=80):
    model.eval()
    encoder_inputs = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)
    encoder_outputs, hidden, cell = model.encoder(encoder_inputs)

    sos_id = sp_ko.bos_id()
    eos_id = sp_ko.eos_id()

    decoder_input = torch.tensor([[sos_id]], dtype=torch.long).to(device)
    decoded_ids = []

    for _ in range(max_output_len):
        output, hidden, cell = model.decoder(decoder_input, encoder_outputs, hidden, cell)
        pred_id = output.argmax(dim=-1).item()
        if pred_id == eos_id:
            break
        decoded_ids.append(pred_id)
        decoder_input = torch.tensor([[pred_id]], dtype=torch.long).to(device)

    # SentencePiece로 디코딩 (서브워드 → 원문 복원)
    return sp_ko.decode(decoded_ids)

def ids_to_en(ids):
    """영어 정수 시퀀스 → 텍스트"""
    return sp_en.decode([int(x) for x in ids if x != 0])

def ids_to_ko(ids):
    """한국어 정수 시퀀스 → 텍스트 (특수토큰 제외)"""
    filtered = [int(x) for x in ids if x != 0 and x != sp_ko.bos_id() and x != sp_ko.eos_id()]
    return sp_ko.decode(filtered)

In [ ]:
# 번역 예시
print("=" * 60)
print("번역 예시")
print("=" * 60)
for seq_index in [3, 50, 100, 300, 1001]:
    input_seq = encoder_input_train[seq_index]
    translated = decode_sequence(input_seq, model)

    print(f"입력: {ids_to_en(encoder_input_train[seq_index])}")
    print(f"정답: {ids_to_ko(decoder_input_train[seq_index])}")
    print(f"번역: {translated}")
    print("-" * 60)

In [ ]:
# BLEU / ROUGE 평가
import evaluate

bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

def calculate_bleu_rouge(model, num_samples=1000):
    model.eval()
    predictions, references = [], []

    print(f"총 {num_samples}개 문장 평가 중...")
    for i in tqdm(range(num_samples)):
        input_seq = encoder_input_test[i]
        target = ids_to_ko(decoder_target_test[i]).strip()
        predicted = decode_sequence(input_seq, model).strip()
        if not predicted:
            predicted = " "
        predictions.append(predicted)
        references.append([target])

    bleu = bleu_metric.compute(predictions=predictions, references=references)
    rouge = rouge_metric.compute(predictions=predictions, references=references)

    print("\n" + "=" * 50)
    print(f"번역 성능 평가 ({num_samples}개)")
    print("=" * 50)
    print(f"BLEU  : {bleu['score']:.2f}")
    print(f"ROUGE-1: {rouge['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge['rougeL']:.4f}")
    print("=" * 50)
    print(f"\n예시 - 정답: {references[0][0]}")
    print(f"       번역: {predictions[0]}")

calculate_bleu_rouge(model, num_samples=1000)